In [1]:
import os
from dotenv import load_dotenv

load_dotenv(".env")

from vectorless_rag import MultimodalClient, PageIndexAPIError, pageindex_tools

client = MultimodalClient(
    index_model=os.environ["PAGEINDEX_INDEX_MODEL"],
    chat_model=os.environ["PAGEINDEX_CHAT_MODEL"],
    storage_path=".pageindex",
    summary_concurrency=10,
)

## 1. Ingest

`submit_document` bersifat sinkron: fungsi baru kembali setelah dokumen selesai diindeks.

In [2]:
FILES = [
    "data/evaluasi_pemdi.pdf",            
    "data/pemdi_merged.pdf",                   
]

docs = {}
for path in FILES:
    doc = client.submit_document(path)
    docs[doc["name"]] = doc["doc_id"]
    print(doc["name"], doc["doc_id"], doc["usage"]["total_tokens"], "token")

[Textract] Mengecek DPI efektif setiap halaman (threshold=1)...
[Textract] Semua 31 halaman lolos cek DPI efektif.
[Textract] Formatted JSON disimpan ke: /tmp/pageindex_mm_zynqyq97/formatted_parsing_result.json


[multimodal] 275a2ac07bc445928ffcfa9f6107e9c5.pdf: Textract/S3/VLM gagal, pakai teks asli (PyPDF2): Upload gambar ke S3 gagal: figure_p29_49c79f3e.jpg
Traceback (most recent call last):
  File "/home/satyananda/vectorless-rag/vectorless_rag/storage/s3/objects.py", line 68, in _put
    upload_fileobj(fh, key, mimetypes.guess_type(path)[0])
    ~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/satyananda/vectorless-rag/vectorless_rag/storage/s3/objects.py", line 32, in upload_fileobj
    get_client().upload_fileobj(
    ~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        fileobj,
        ^^^^^^^^
    ...<2 lines>...
        ExtraArgs={"ContentType": content_type or "application/octet-stream"},
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/home/satyananda/vectorless-rag/.venv/lib/python3.13/site-packages/botocore/context.py", line 123, in wrapper
    return func(*args, **kwargs)
  File "/home/satyananda/vectorless-rag/.venv/lib/pyt

evaluasi_pemdi.pdf pi-a591e15f21d14beaa8a434562e00f2ab 155370 token
[Textract] Mengecek DPI efektif setiap halaman (threshold=1)...
[Textract] Semua 13 halaman lolos cek DPI efektif.
[Textract] Formatted JSON disimpan ke: /tmp/pageindex_mm_fwz7m097/formatted_parsing_result.json
pemdi_merged.pdf pi-f49c1a34710a40b89b8112e314fcc6b4 45921 token


In [ ]:
for d in client.list_documents()["documents"]:
    print(d["id"], "|", d["name"], "|", d.get("description"))

## 2. Retrieve (tanpa LLM)

Fungsi-fungsi ini adalah data yang juga dibaca tool agent:
- `get_document_structure`: tree berisi judul, rentang halaman, dan summary.
- `get_page_content`: teks halaman. Untuk halaman yang lewat Textract, isinya markdown dengan tabel dan deskripsi gambar.

In [ ]:
renstra = "Rancangan_Akhir_Renstra_Diskominfo_Jatim_2025-2029.pdf"
doc_id = docs[renstra]

def print_tree(nodes, depth=0):
    for node in nodes:
        print(f"{'  ' * depth}- [{node['start_index']}-{node['end_index']}] {node['title']}")
        print_tree(node.get("nodes") or [], depth + 1)

print_tree(client.get_document_structure(doc_id))

In [ ]:
for page in client.get_page_content(doc_id, "1"):
    print(f"--- halaman {page['page_index']} ---")
    print(page["markdown"][:2000])

## 3. Tools untuk agent

`pageindex_tools(client)` memberi 4 LangGraph tool (`browse_documents`, `get_document`, `get_document_structure`, `get_page_content`) untuk dipasang di agent. `doc_ids` membatasi dokumen yang boleh dibaca. Setiap halaman dari `get_page_content` membawa `citation` (`file` + `page`, atau `file` + `sheet` untuk `.xlsx`).

In [ ]:
tools = {t.name: t for t in pageindex_tools(client)}
print(tools["browse_documents"].invoke({}))

In [ ]:
print(tools["get_document_structure"].invoke({"doc_name": renstra}))

In [ ]:
# Dokumen office (dikonversi ke PDF, halaman scan dibaca lewat OCR)
docx = "DOC_L1-010_scan_halaman_tengah_renja_jatim.docx"
print(tools["get_page_content"].invoke({"doc_name": docx, "pages": "1"}))

In [ ]:
# Dibatasi ke Renstra saja: dokumen lain ditolak
scoped = {t.name: t for t in pageindex_tools(client, doc_ids=[docs[renstra]])}
print(scoped["get_page_content"].invoke({"doc_name": docx, "pages": "1"}))

## 4. Delete

Penghapusan bersifat permanen: folder dokumen dan entry-nya di manifest ikut terhapus.

In [ ]:
print(client.delete_document(doc_id))

try:
    client.get_document(doc_id)
except PageIndexAPIError as e:
    print("get_document ->", e)

In [ ]:
# Hapus semua dokumen di library ini
for d in client.list_documents(limit=1000)["documents"]:
    client.delete_document(d["id"])
print("tersisa:", client.list_documents()["total"])